In [ ]:
from pathlib import Path

import os
import requests
import time 
import sys

# For saving it as a csv file 
import pandas as pd 
# For numeric analysis 
import numpy as np 
# For time analysis functions 
from datetime import datetime, timedelta


# Add the src directory to the system path
src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path: sys.path.append(src_path)
# Import your fetch function
from utils import *

# Lösung für Notebooks: 
# In Jupyter: Arbeitsverzeichnis (z. B. Projektordner)
project_root = Path().resolve().parents[0]
sys.path.append(str(project_root))
# Path wo die roh daten gespeichert sind 
path_data_raw       = project_root / "data" / "raw"
# Path wo die processed data gespeichert wird 
path_data_processed = project_root / "data" / "processed"
# Path wo die processed data gespeichert wird 
path_plots           = project_root / "reports" / "plots"

print(project_root)
print(path_data_processed)
print(path_data_raw)

# Pfade sicherstellen für die process data files und fuer die plot files im reports Folder 
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("reports/plots").mkdir(parents=True, exist_ok=True)



/Users/tillo/Repositorys/stock-price-rnn
/Users/tillo/Repositorys/stock-price-rnn/data/processed
/Users/tillo/Repositorys/stock-price-rnn/data/raw


In [2]:
# Finde alle CSV-Dateien
csv_files = list_csv_files(path_data_processed)

#print(data_folder)
#print(csv_files)

# Lade sie in ein Dictionary mit DataFrames
dfs = load_csvs_as_dfs(csv_files, path_data_processed)

# Zugriff auf einen bestimmten DataFrame:
print(dfs.keys())  # z. B. df_apple_stock, df_dax_2020 etc.

# Erstellen einer liste der keys der aktien 
keys_list = list(dfs.keys())
print(keys_list[0])


dict_keys(['df_df_aapl_last_1000_days', 'df_df_msft_last_1000_days', 'df_df_mbg.dex_last_1000_days', 'df_df_googl_last_1000_days'])
df_df_aapl_last_1000_days


In [4]:
# Hier erstmal zwischen gespiechert die funktionen: 

# Analyse von den jeweils einzelnen aktien 
def analyze_stock(name, df):
    return {
        'Stock': name,
        'PositiveTrend6M': detect_positive_trend(df),
        'Last3DaysDown': last_3_days_downward(df),
        '30%BelowLastPeak': below_last_peak(df, threshold=0.3),
        'Seasonality': has_seasonality(df)
    }

# einzelne analyse funktionen 
def detect_positive_trend(df, months=6):
    if len(df) < 2:
        return False
    df['Date'] = pd.to_datetime(df['Date'])
    recent_df = df[df['Date'] >= df['Date'].max() - pd.DateOffset(months=months)]
    if len(recent_df) < 2:
        return False
    x = np.arange(len(recent_df))
    y = recent_df['close'].values
    slope = np.polyfit(x, y, 1)[0]
    return slope > 0

def last_3_days_downward(df):
    df_sorted = df.sort_values('Date')
    recent = df_sorted.tail(3)
    return all(recent['close'].diff().dropna() < 0)

def below_last_peak(df, threshold=0.3):
    peaks = df[df['is_peak'] == True]
    if peaks.empty:
        return False
    last_peak_value = peaks['close'].iloc[-1]
    current_value = df['close'].iloc[-1]
    return current_value < (1 - threshold) * last_peak_value

def has_seasonality(df):
    # Naiv: prüfe, ob Abstand zwischen zwei größten Peaks ca. gleich ist
    peaks = df[df['is_peak']]
    if len(peaks) < 3:
        return False
    peak_dates = pd.to_datetime(peaks['Date'])
    diffs = (peak_dates - peak_dates.shift()).dropna().dt.days
    if diffs.empty:
        return False
    std = diffs.std()
    return std < 20  # heuristischer Grenzwert


In [5]:
results = []

for name, df in dfs.items():
    # Stelle sicher, dass alle benötigten Spalten vorhanden sind
    if 'Date' in df.columns and 'close' in df.columns and 'is_peak' in df.columns:
        result = analyze_stock(name, df)
        results.append(result)
    else:
        print(f"Skipping {name}, missing required columns.")

# Ergebnis als Übersichtstabelle
overview_df = pd.DataFrame(results)

overview_df.head() 

Skipping df_df_aapl_last_1000_days, missing required columns.
Skipping df_df_msft_last_1000_days, missing required columns.
Skipping df_df_mbg.dex_last_1000_days, missing required columns.
Skipping df_df_googl_last_1000_days, missing required columns.


""
